# 🔔 Akıllı Hatırlatma Sistemi

**Amaç:** Çiftçiye zamanı gelen bakım işlerini otomatik hatırlatan bir sistem
kurmak. Örneğin aşı zamanı, tahmini doğum tarihi, ilaç bekleme süresinin bitişi.

**Yöntem:** Tarih tabanlı kural mantığı. Her hayvanın kayıtlı tarihlerine bakıp,
bugüne göre "yaklaşan" veya "geçmiş" görevleri tespit eder.

**Not:** Bu bölüm yapay zeka değil, tarih/kural tabanlı bir sistemdir; ancak
uygulamanın en pratik ve günlük kullanılan özelliklerinden biridir.

In [1]:
from datetime import datetime, timedelta

# Bugünün tarihi
bugun = datetime.now()

# Örnek hayvan kayıtları (gerçekte bu veriler veritabanından gelecek)
hayvanlar = [
    {"kupe_no": "TR-001", "ad": "Sarıkız",
     "son_asi": datetime(2025, 2, 1), "asi_periyodu_gun": 180,
     "gebelik_tarihi": datetime(2025, 1, 15),
     "ilac_bekleme_bitis": None},

    {"kupe_no": "TR-002", "ad": "Benekli",
     "son_asi": datetime(2025, 7, 20), "asi_periyodu_gun": 180,
     "gebelik_tarihi": None,
     "ilac_bekleme_bitis": datetime(2025, 8, 5)},

    {"kupe_no": "TR-003", "ad": "Karakaş",
     "son_asi": datetime(2025, 6, 10), "asi_periyodu_gun": 180,
     "gebelik_tarihi": datetime(2024, 12, 1),
     "ilac_bekleme_bitis": None},
]

print("Bugün:", bugun.strftime("%d.%m.%Y"))
print("Kayıtlı hayvan sayısı:", len(hayvanlar))

Bugün: 03.08.2026
Kayıtlı hayvan sayısı: 3


In [2]:
def hatirlatmalari_getir(hayvan):
    """Bir hayvan için zamanı gelen/yaklaşan hatırlatmaları üretir."""
    hatirlatmalar = []

    # 1. AŞI kontrolü
    if hayvan["son_asi"]:
        sonraki_asi = hayvan["son_asi"] + timedelta(days=hayvan["asi_periyodu_gun"])
        kalan_gun = (sonraki_asi - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"💉 AŞI GECİKMİŞ! ({-kalan_gun} gün önce yapılmalıydı)")
        elif kalan_gun <= 15:
            hatirlatmalar.append(f"💉 Aşı zamanı yaklaşıyor ({kalan_gun} gün kaldı)")

    # 2. DOĞUM kontrolü (gebelik ~283 gün)
    if hayvan["gebelik_tarihi"]:
        tahmini_dogum = hayvan["gebelik_tarihi"] + timedelta(days=283)
        kalan_gun = (tahmini_dogum - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini {-kalan_gun} gün önceydi, kontrol edin)")
        elif kalan_gun <= 15:
            hatirlatmalar.append(f"🐄 Doğum yaklaşıyor ({kalan_gun} gün kaldı) — doğum bölmesini hazırlayın")

    # 3. İLAÇ BEKLEME kontrolü
    if hayvan["ilac_bekleme_bitis"]:
        kalan_gun = (hayvan["ilac_bekleme_bitis"] - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"💊 İlaç bekleme süresi bitti — süt/et artık kullanılabilir")
        elif kalan_gun >= 0:
            hatirlatmalar.append(f"⛔ İlaç bekleme süresi devam ediyor ({kalan_gun} gün kaldı) — süt/et KULLANMAYIN")

    return hatirlatmalar


# Tüm hayvanlar için hatırlatmaları göster
print("=" * 55)
print(f"📅 HATIRLATMALAR — {bugun.strftime('%d.%m.%Y')}")
print("=" * 55)

for hayvan in hayvanlar:
    mesajlar = hatirlatmalari_getir(hayvan)
    if mesajlar:
        print(f"\n🐮 {hayvan['ad']} ({hayvan['kupe_no']})")
        for m in mesajlar:
            print(f"   {m}")

print("\n" + "=" * 55)

📅 HATIRLATMALAR — 03.08.2026

🐮 Sarıkız (TR-001)
   💉 AŞI GECİKMİŞ! (369 gün önce yapılmalıydı)
   🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini 283 gün önceydi, kontrol edin)

🐮 Benekli (TR-002)
   💉 AŞI GECİKMİŞ! (200 gün önce yapılmalıydı)
   💊 İlaç bekleme süresi bitti — süt/et artık kullanılabilir

🐮 Karakaş (TR-003)
   💉 AŞI GECİKMİŞ! (240 gün önce yapılmalıydı)
   🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini 328 gün önceydi, kontrol edin)



In [3]:
# Yaklaşan durumu test etmek için: bugüne yakın tarihli bir hayvan
test_hayvan = {
    "kupe_no": "TR-004", "ad": "Yıldız",
    "son_asi": bugun - timedelta(days=170),   # 170 gün önce aşı (periyot 180, yani 10 gün kaldı)
    "asi_periyodu_gun": 180,
    "gebelik_tarihi": bugun - timedelta(days=275),  # 275 gün önce gebe (283'e 8 gün kaldı)
    "ilac_bekleme_bitis": None
}

print(f"🐮 {test_hayvan['ad']} ({test_hayvan['kupe_no']})")
for m in hatirlatmalari_getir(test_hayvan):
    print(f"   {m}")

🐮 Yıldız (TR-004)
   💉 Aşı zamanı yaklaşıyor (10 gün kaldı)
   🐄 Doğum yaklaşıyor (8 gün kaldı) — doğum bölmesini hazırlayın


## 🎯 Kapanış — Akıllı Hatırlatma Sistemi

Bu notebook'ta, çiftçiye zamanı gelen bakım işlerini otomatik hatırlatan tarih
tabanlı bir sistem kurduk.

**Yapılanlar**
- Her hayvan için aşı, doğum ve ilaç bekleme tarihlerini takip etme
- `datetime` ve `timedelta` ile "kaç gün kaldı / kaç gün geçti" hesabı
- Üç tür hatırlatma: aşı zamanı (periyoda göre), tahmini doğum (283 günlük
  gebelik hesabı), ilaç bekleme süresi (gıda güvenliği için)
- Hem geçmiş/geciken hem de yaklaşan görevleri tespit etme

**Sonuç**
Sistem, her hayvanın kayıtlı tarihlerine bakarak "aşı gecikmiş", "doğum yaklaşıyor
(8 gün kaldı)", "ilaç bekleme süresi devam ediyor" gibi net hatırlatmalar üretiyor.
Bu, uygulamanın günlük kullanılan, pratik özelliklerinden biridir.

**Not:** Şimdilik örnek verilerle çalışıyor; gerçek uygulamada bu tarihler
veritabanından (sağlık kaydı ve gebelik tablolarından) gelecektir.